# C-MAPSS model experiment runner

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((directory for directory in (current_directory, *current_directory.parents)if (directory / 'pyproject.toml').is_file()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root')

CODE_ROOT = PROJECT_ROOT / 'code'

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

os.chdir(PROJECT_ROOT)
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
os.environ.setdefault('MLFLOW_TRACKING_URI', f'file:{PROJECT_ROOT / "mlruns"}')
os.environ.setdefault('CMAPSS_MLFLOW_EXPERIMENT', 'cmapss-preprocessing')
os.environ.setdefault('CMAPSS_MLFLOW_TRAINING_EXPERIMENT', 'cmapss-training')

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/aanchal/nasa_c_mapss


In [5]:
from src.training.lstm import train_lstm
from src.build_spark import spark_session_context
from src.training.tree_models import train_random_forest
from src.data_processing.preprocessing import run_subset_preprocessing
from src.training.tree_models.run_tabular_training import run_training
from src.training.evaluation.test_evaluation import evaluate_test_data

from src.data_processing import build_endpoint_sequences
from src.tracking import load_training_feature_columns, load_training_model
from src.training.evaluation.engine_endpoint_evaluation import (evaluate_prediction_diagnostics,
                                                                prepare_pseudo_test_validation,
                                                                select_pseudo_test_endpoints)
from src.training.evaluation.model_evaluation_metrics import evaluate_predictions

In [6]:
SUBSETS = ('FD001', 'FD002', 'FD003', 'FD004')
RAW_DATA_DIR = PROJECT_ROOT / 'Data' / 'CMAPSSData'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'Data' / 'processed'
RUL_CAP = 125

RUN_TESTS = True
RUN_TRAINING = True

RUN_TEMPORAL_PREPROCESSING = False

In [7]:
BASELINE_PREPROCESSING_RUN_IDS = {
    'FD001': '208b1c6c99264e74a42f35df2edfd4dd',
    'FD002': '1030d9d2baf7491e901e85ed69d5655f',
    'FD003': '61dd784103e24f408aea4486cb982a35',
    'FD004': '432ff4df92c7451b97e52832049b5a70',
}

preprocessing_run_ids = {(subset, 'baseline'): run_id for subset, run_id in BASELINE_PREPROCESSING_RUN_IDS.items()}

preprocessing_results = [{'subset': subset, 
                          'feature_set': 'baseline', 
                          'preprocessing_run_id': run_id, 
                          'source': 'existing'}
                         for subset, run_id in BASELINE_PREPROCESSING_RUN_IDS.items()]


In [8]:
TEMPORAL_PREPROCESSING_RUN_IDS = {
    'FD001': '20db40e34e4b42f392e931eab4ae1b7d',
    'FD002': '324277ebf695472886eaf129724e9833',
    'FD003': '246b7b1ac64d455298f08db0b0abd11d',
    'FD004': '0bd8c9d6df084181b95c5048faa9ad5f',
}

preprocessing_run_ids.update({(subset, 'temporal'): run_id for subset, run_id in TEMPORAL_PREPROCESSING_RUN_IDS.items()})

preprocessing_results.extend({'subset': subset,
                              'feature_set': 'temporal',
                              'preprocessing_run_id': run_id,
                              'source': 'existing'}
                             for subset, run_id in TEMPORAL_PREPROCESSING_RUN_IDS.items())

if RUN_TEMPORAL_PREPROCESSING:
    for subset in SUBSETS:
        print(f'\n[{subset}] preprocessing temporal features...', flush=True)
        
        with spark_session_context(app_name=f'cmapss-{subset}-temporal-notebook') as spark:
            
            result = run_subset_preprocessing(spark=spark, 
            subset=subset, 
            raw_data_dir=RAW_DATA_DIR, 
            output_dir=PROCESSED_DATA_DIR, 
            include_temporal_features=True)
        
        preprocessing_run_ids[(subset, 'temporal')] = result.run_id
        
        preprocessing_results.append({'subset': subset,
                                      'feature_set': 'temporal',
                                      'preprocessing_run_id': result.run_id,
                                      'source': 'created',
                                      'feature_count': result.feature_count,
                                      'train_rows': result.train_row_count})

In [9]:
pd.DataFrame(preprocessing_results)

,subset,feature_set,preprocessing_run_id,source
0,FD001,baseline,208b1c6c99264e74a42f35df2edfd4dd,existing
1,FD002,baseline,1030d9d2baf7491e901e85ed69d5655f,existing
2,FD003,baseline,61dd784103e24f408aea4486cb982a35,existing
3,FD004,baseline,432ff4df92c7451b97e52832049b5a70,existing
4,FD001,temporal,20db40e34e4b42f392e931eab4ae1b7d,existing
5,FD002,temporal,324277ebf695472886eaf129724e9833,existing
6,FD003,temporal,246b7b1ac64d455298f08db0b0abd11d,existing
7,FD004,temporal,0bd8c9d6df084181b95c5048faa9ad5f,existing


In [ ]:
training_results = []

if RUN_TRAINING:
    required_preprocessing_runs = {(subset, feature_set) for subset in SUBSETS for feature_set in ('baseline', 'temporal')}
    
    missing_runs = required_preprocessing_runs - preprocessing_run_ids.keys()
    if missing_runs:
        raise RuntimeError(f'Missing preprocessing runs: {sorted(missing_runs)}')

    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]

            print(f'\n[{subset}] training RF with {feature_set} features...', flush=True)
            
            rf_run_id = train_random_forest(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                            processed_data_dir=PROCESSED_DATA_DIR, rul_cap=RUL_CAP)
            
            rf_test_metrics = evaluate_test_data(subset_id=subset, training_run_id=rf_run_id, 
                                                 processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)
            
            training_results.append({
                    'subset': subset,
                    'model': 'random_forest',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': rf_run_id,
                    **{f'test_{name}': value for name, value in rf_test_metrics.items()}})

            print(f'\n[{subset}] training XGBoost with {feature_set} features...', flush=True)
            
            xgboost_result = run_training( model_type='xgboost', subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                          processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)
            
            training_results.append({
                    'subset': subset,
                    'model': 'xgboost',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': xgboost_result['training_run_id'],
                    **{f'test_{name}': value for name, value in xgboost_result['test_metrics'].items()}})

    for subset in SUBSETS:        
        preprocessing_run_id = preprocessing_run_ids[(subset, 'baseline')]
        
        print(f'\n[{subset}] training capped-target LSTM...', flush=True)
        
        lstm_result = train_lstm(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                                 processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR, rul_cap=RUL_CAP)
        
        training_results.append({
                'subset': subset,
                'model': 'lstm',
                'feature_set': 'base_sequence',
                'preprocessing_run_id': preprocessing_run_id,
                'training_run_id': lstm_result['training_run_id'],
                **{f'test_{name}': value for name, value in lstm_result['test_metrics'].items()}})

results = pd.DataFrame(training_results)

results

In [ ]:
comparison_columns = [
    'test_mae',
    'test_rmse',
    'test_nasa_score',
    'test_bias',
    'test_late_prediction_rate',
    'test_worst_positive_error',
    'test_worst_negative_error',
    'test_largest_nasa_contribution',
    'test_top_3_nasa_contribution',
]

results.set_index(['subset', 'model', 'feature_set'])[comparison_columns].round(3)

In [ ]:
temporal_lstm_results = []

for subset in SUBSETS:
    preprocessing_run_id = TEMPORAL_PREPROCESSING_RUN_IDS[subset]
    print(f'\n[{subset}] training capped-target LSTM with temporal features...', flush=True)
    
    result = train_lstm(subset_id=subset, preprocessing_run_id=preprocessing_run_id, 
                        processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR, 
                        rul_cap=RUL_CAP, feature_set='temporal_sequence')
    
    temporal_lstm_results.append({
            'subset': subset,
            'model': 'lstm',
            'feature_set': 'temporal_sequence',
            'preprocessing_run_id': preprocessing_run_id,
            'training_run_id': result['training_run_id'],
            **{f'test_{name}': value for name, value in result['test_metrics'].items()}})

pd.DataFrame(temporal_lstm_results)

In [13]:
ROBUSTNESS_SEEDS = tuple(range(42, 52))

ROBUSTNESS_CANDIDATES = (
    {'subset': 'FD001', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': 'e9ee327ea7814c0bafc8179e188fcba7'},
    {'subset': 'FD001', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD001'], 'training_run_id': 'd37624ea98f5459db673a69c5dc56951'},
    {'subset': 'FD002', 'candidate': 'temporal_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '470f26004ed0462abc492a62872ce600'},
    {'subset': 'FD002', 'candidate': 'temporal_xgboost', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD002'], 'training_run_id': '1b7584f00aad4746a59ff0103f2448c3'},
    {'subset': 'FD003', 'candidate': 'base_lstm', 'role': 'champion', 'model_family': 'lstm', 'preprocessing_run_id': BASELINE_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': '1ad1630ccad347b99abc71ed1317566b'},
    {'subset': 'FD003', 'candidate': 'temporal_lstm', 'role': 'competitor', 'model_family': 'lstm', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD003'], 'training_run_id': '9a1fce6e99cd401a9891a8ca210c629a'},
    {'subset': 'FD004', 'candidate': 'temporal_xgboost', 'role': 'champion', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '1b90ed34b6964e58bcef186c090cfc52'},
    {'subset': 'FD004', 'candidate': 'temporal_random_forest', 'role': 'competitor', 'model_family': 'tabular', 'preprocessing_run_id': TEMPORAL_PREPROCESSING_RUN_IDS['FD004'], 'training_run_id': '9765b1d0d6e34ec49360fa8b9bbeb09f'},
)

In [14]:
robustness_results = []

for candidate in ROBUSTNESS_CANDIDATES:
    validation_path = (PROCESSED_DATA_DIR / candidate['subset'] / candidate['preprocessing_run_id'] / 'validation')
    validation_dataframe = pd.read_parquet(validation_path)
    
    feature_columns = load_training_feature_columns(candidate['training_run_id'])
    validation_dataframe[feature_columns] = (validation_dataframe[feature_columns].fillna(0).astype(np.float32))
    
    model = load_training_model(candidate['training_run_id'])

    for seed in ROBUSTNESS_SEEDS:
        if candidate['model_family'] == 'lstm':
            metadata = select_pseudo_test_endpoints(validation_dataframe, seed=seed)
            sequences = build_endpoint_sequences(validation_dataframe, metadata, feature_columns, sequence_length=30)
            
            predictions = np.asarray([np.asarray(model.predict(sequence[np.newaxis, ...])).reshape(-1)[0] for sequence in sequences])
            targets = metadata['RUL']
            
        else:
            features, targets, metadata = prepare_pseudo_test_validation(validation_dataframe, feature_columns, seed=seed)
            predictions = model.predict(features)

        metrics = evaluate_predictions(targets, predictions)
        _, tail_metrics = evaluate_prediction_diagnostics(metadata, predictions)
        robustness_results.append({**candidate, 'seed': seed, **metrics, **tail_metrics})

robustness_by_seed = pd.DataFrame(robustness_results)
robustness_summary = (robustness_by_seed.groupby(['subset', 'candidate', 'role', 'preprocessing_run_id', 'training_run_id'], as_index=False)
                      .agg(mean_mae=('mae', 'mean'), 
                      median_mae=('mae', 'median'), 
                      mean_rmse=('rmse', 'mean'), 
                      median_rmse=('rmse', 'median'), 
                      mean_nasa_score=('nasa_score', 'mean'), 
                      median_nasa_score=('nasa_score', 'median'), 
                      worst_seed_nasa_score=('nasa_score', 'max'), 
                      worst_positive_error=('worst_positive_error', 'max')))

champion_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'champion', ['subset', 'seed', 'nasa_score']
                                         ].rename(columns={'nasa_score': 'champion_nasa_score'})

competitor_scores = robustness_by_seed.loc[robustness_by_seed['role'] == 'competitor', ['subset', 'seed', 'nasa_score']
                                           ].rename(columns={'nasa_score': 'competitor_nasa_score'})

win_rates = champion_scores.merge(competitor_scores, on=['subset', 'seed'])
win_rates = (win_rates.assign(champion_win=lambda frame: frame['champion_nasa_score'] < frame['competitor_nasa_score'])
             .groupby('subset', as_index=False)['champion_win']
             .mean()
             .rename(columns={'champion_win': 'champion_nasa_win_rate'}))

robustness_summary.merge(win_rates, on='subset', how='left').round(3)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_job

,subset,candidate,role,preprocessing_run_id,training_run_id,mean_mae,median_mae,mean_rmse,median_rmse,mean_nasa_score,median_nasa_score,worst_seed_nasa_score,worst_positive_error,champion_nasa_win_rate
0,FD001,temporal_lstm,champion,20db40e34e4b42f392e931eab4ae1b7d,e9ee327ea7814c0bafc8179e188fcba7,11.257,10.367,16.967,17.441,155.897,173.238,239.123,49.728,0.3
1,FD001,temporal_xgboost,competitor,20db40e34e4b42f392e931eab4ae1b7d,d37624ea98f5459db673a69c5dc56951,11.213,11.463,15.797,16.650,109.719,102.083,192.131,43.889,0.3
2,FD002,temporal_lstm,champion,324277ebf695472886eaf129724e9833,470f26004ed0462abc492a62872ce600,12.234,11.939,16.837,16.624,366.415,355.712,571.830,51.757,0.9
3,FD002,temporal_xgboost,competitor,324277ebf695472886eaf129724e9833,1b7584f00aad4746a59ff0103f2448c3,14.426,14.232,19.733,19.981,667.395,666.531,1005.070,56.423,0.9
4,FD003,base_lstm,champion,61dd784103e24f408aea4486cb982a35,1ad1630ccad347b99abc71ed1317566b,10.975,10.803,15.818,15.393,130.956,72.491,443.369,53.596,0.7
5,FD003,temporal_lstm,competitor,246b7b1ac64d455298f08db0b0abd11d,9a1fce6e99cd401a9891a8ca210c629a,12.248,12.380,17.604,18.239,211.120,176.033,573.923,59.067,0.7
6,FD004,temporal_random_forest,competitor,0bd8c9d6df084181b95c5048faa9ad5f,9765b1d0d6e34ec49360fa8b9bbeb09f,15.602,15.171,21.243,20.898,907.170,832.552,1758.322,70.480,0.7
7,FD004,temporal_xgboost,champion,0bd8c9d6df084181b95c5048faa9ad5f,1b90ed34b6964e58bcef186c090cfc52,15.513,15.188,21.063,20.516,762.804,673.860,1716.787,67.194,0.7


## LightGBM training

In [10]:
lightgbm_results = []

if RUN_TRAINING:
    for feature_set in ('baseline', 'temporal'):
        for subset in SUBSETS:
            preprocessing_run_id = preprocessing_run_ids[(subset, feature_set)]
            print(f'\n[{subset}] training LightGBM with {feature_set} features...', flush=True)

            result = run_training(model_type='lightgbm', subset_id=subset,
                                  preprocessing_run_id=preprocessing_run_id,
                                  processed_data_dir=PROCESSED_DATA_DIR, raw_data_dir=RAW_DATA_DIR)

            lightgbm_results.append({
                    'subset': subset,
                    'model': 'lightgbm',
                    'feature_set': feature_set,
                    'preprocessing_run_id': preprocessing_run_id,
                    'training_run_id': result['training_run_id'],
                    **{f'test_{name}': value for name, value in result['test_metrics'].items()}})

pd.DataFrame(lightgbm_results)


[FD001] training LightGBM with baseline features...
[FD001] starting model training...
[FD001] fitting lightgbm on 16,260 rows and 15baseline features with target=capped_rul_125...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.059150 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2665
[LightGBM] [Info] Number of data points in the train set: 16260, number of used features: 15
[LightGBM] [Info] Start training from score 86.254613


2026/08/24 17:31:55 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:32:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:32:12 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/24 17:32:13 INFO mlflow.utils.uv_utils: Exported 95 dependencies via uv
2026/08/24 17:32:13 INFO mlflow.utils.environment: Successfully exported 95 requirements from uv project. Skipping package capture based inference.
2026/08/24 17:32:13 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/24 17:32:13 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/24 17:32:15 WARNING mlflow.utils.environment: Fa

[FD001] evaluating the test set...

[FD002] training LightGBM with baseline features...
[FD002] starting model training...
[FD002] fitting lightgbm on 43,289 rows and 21baseline features with target=capped_rul_125...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3406
[LightGBM] [Info] Number of data points in the train set: 43289, number of used features: 17
[LightGBM] [Info] Start training from score 87.161288


2026/08/24 17:32:40 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:32:41 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:32:41 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/24 17:32:42 INFO mlflow.utils.uv_utils: Exported 95 dependencies via uv
2026/08/24 17:32:42 INFO mlflow.utils.environment: Successfully exported 95 requirements from uv project. Skipping package capture based inference.
2026/08/24 17:32:42 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/24 17:32:42 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/24 17:32:42 WARNING mlflow.utils.environment: Fa

[FD002] evaluating the test set...

[FD003] training LightGBM with baseline features...
[FD003] starting model training...
[FD003] fitting lightgbm on 19,979 rows and 16baseline features with target=capped_rul_125...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2955
[LightGBM] [Info] Number of data points in the train set: 19979, number of used features: 16
[LightGBM] [Info] Start training from score 93.466890


...

2026/08/24 17:34:28 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:34:28 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:34:28 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/24 17:34:29 INFO mlflow.utils.uv_utils: Exported 95 dependencies via uv
2026/08/24 17:34:29 INFO mlflow.utils.environment: Successfully exported 95 requirements from uv project. Skipping package capture based inference.
2026/08/24 17:34:29 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/24 17:34:29 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/24 17:34:29 WARNING mlflow.utils.environment: Fa

[FD003] evaluating the test set...

[FD004] training LightGBM with temporal features...
[FD004] starting model training...
[FD004] fitting lightgbm on 48,733 rows and 315temporal features with target=capped_rul_125...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.160079 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61247
[LightGBM] [Info] Number of data points in the train set: 48733, number of used features: 295
[LightGBM] [Info] Start training from score 92.842632


2026/08/24 17:35:10 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:35:10 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/aanchal/nasa_c_mapss
2026/08/24 17:35:10 INFO mlflow.utils.environment: Detected uv project at /home/aanchal/nasa_c_mapss. Attempting to export requirements via 'uv export'.
2026/08/24 17:35:10 INFO mlflow.utils.uv_utils: Exported 95 dependencies via uv
2026/08/24 17:35:10 INFO mlflow.utils.environment: Successfully exported 95 requirements from uv project. Skipping package capture based inference.
2026/08/24 17:35:10 INFO mlflow.utils.uv_utils: Extracted 1 private index URL(s) from uv.lock
2026/08/24 17:35:10 WARNING mlflow.utils.uv_utils: Private package indexes detected in uv lockfile. Ensure credentials are available at model load time via UV_INDEX_* environment variables or .netrc file.
2026/08/24 17:35:10 WARNING mlflow.utils.environment: Fa

[FD004] evaluating the test set...


,subset,model,feature_set,preprocessing_run_id,training_run_id,test_rmse,test_mae,test_nasa_score,test_bias,test_late_prediction_rate,test_worst_positive_error,test_worst_negative_error,test_largest_nasa_contribution,test_top_3_nasa_contribution
0,FD001,lightgbm,baseline,208b1c6c99264e74a42f35df2edfd4dd,5b31c76a2f4d4b32b15fd43b8def9bf2,18.277638,13.413375,929.629565,2.242695,0.530000,51.842126,-39.694661,177.432889,482.361325
1,FD002,lightgbm,baseline,1030d9d2baf7491e901e85ed69d5655f,2bec827bcfb4444f9373f54b8db3cdc3,29.732782,20.717718,14103.526513,-6.411098,0.482625,76.274476,-96.468144,2052.801278,5089.051127
2,FD003,lightgbm,baseline,61dd784103e24f408aea4486cb982a35,c67790b4124c4d6a83e7cf9234d9b8f4,21.681080,16.225717,2274.389357,6.273901,0.640000,65.675315,-34.432175,710.611040,1198.680719
3,FD004,lightgbm,baseline,432ff4df92c7451b97e52832049b5a70,9d032f47ed7a4294882d834cc638e884,30.115638,22.787732,8566.757286,-7.309637,0.475806,76.100306,-81.914913,2017.339797,2949.527074
4,FD001,lightgbm,temporal,20db40e34e4b42f392e931eab4ae1b7d,59bd18a1f3d24bbf834efe5a03f766a8,16.709698,12.712610,534.070055,1.139420,0.580000,44.690825,-45.261849,86.276612,224.426052
5,FD002,lightgbm,temporal,324277ebf695472886eaf129724e9833,52e208d9d2084fad914e377b70ba392c,28.046711,18.703862,13509.042923,-7.846332,0.413127,50.747823,-105.387344,3315.677363,7096.025745
6,FD003,lightgbm,temporal,246b7b1ac64d455298f08db0b0abd11d,2bfd2b85f479425e8e003cce846d8224,17.153710,12.737968,800.655021,2.211928,0.520000,58.865601,-51.315585,359.164213,446.041798
7,FD004,lightgbm,temporal,0bd8c9d6df084181b95c5048faa9ad5f,625296af352d4439b483298c9b268d4a,27.687826,20.358149,5090.603006,-7.557282,0.463710,51.653163,-76.283342,352.523284,915.610904


In [18]:
import mlflow
import mlflow.lightgbm as mlflow_lightgbm
import mlflow.sklearn as mlflow_sklearn
import mlflow.xgboost as mlflow_xgboost
from src.tracking import configure_mlflow

feature_importance_results = []
model_names = {'RandomForestRegressor': 'random_forest',
               'XGBRegressor': 'xgboost',
               'LGBMRegressor': 'lightgbm'}

model_loaders = {'random_forest': mlflow_sklearn.load_model,
                 'xgboost': mlflow_xgboost.load_model,
                 'lightgbm': mlflow_lightgbm.load_model}

configure_mlflow()

tracked_runs = mlflow.search_runs(
        experiment_names=[os.getenv('CMAPSS_MLFLOW_TRAINING_EXPERIMENT', 'cmapss-training')],
        output_format='pandas')

latest_model_runs = (tracked_runs[
        tracked_runs['status'].eq('FINISHED')
        & tracked_runs['params.model_type'].isin(model_names)
        & tracked_runs['tags.feature_set'].isin(('baseline', 'temporal'))]
        .sort_values('start_time', ascending=False)
        .drop_duplicates(['params.subset', 'params.model_type', 'tags.feature_set']))


for result in latest_model_runs.to_dict('records'):
        
    result['model'] = model_names[result['params.model_type']]
    result['training_run_id'] = result['run_id']
    result['subset'] = result['params.subset']
    result['feature_set'] = result['tags.feature_set']
    
    model = model_loaders[result['model']](f"runs:/{result['training_run_id']}/model")
    
    feature_columns = load_training_feature_columns(result['training_run_id'])

    feature_importance_results.extend(
            {'subset': result['subset'], 'model': result['model'], 'feature_set': result['feature_set'],
             'training_run_id': result['training_run_id'],
             'feature': feature, 'importance': importance}            
            for feature, importance in zip(feature_columns, model.feature_importances_))

feature_importance_results = pd.DataFrame(feature_importance_results)

feature_importance_results = (feature_importance_results
        .sort_values(['subset', 'model', 'feature_set', 'importance'], ascending=[True, True, True, False])
        .groupby(['subset', 'model', 'feature_set'], as_index=False).head(20)
        .reset_index(drop=True))

In [24]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD001")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

,subset,model,feature_set,training_run_id,feature,importance
85,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_4_ewma,0.358479
86,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_4_rolling_mean_10,0.115354
87,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_11_rolling_mean_10,0.086455
88,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_4_rolling_mean_5,0.056420
89,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_15_rolling_mean_10,0.031279
90,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_11_rolling_mean_5,0.026085
91,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_9_rolling_mean_5,0.025092
92,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_11_ewma,0.022094
93,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_9_ewma,0.020235
94,FD001,xgboost,temporal,d37624ea98f5459db673a69c5dc56951,sensor_2_rolling_mean_10,0.017928


In [25]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD002")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

,subset,model,feature_set,training_run_id,feature,importance
205,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_11_rolling_mean_10,0.212376
206,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_11_ewma,0.196777
207,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_4_ewma,0.142491
208,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_11_rolling_mean_5,0.082343
209,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_9_rolling_mean_5,0.025876
210,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_4_rolling_mean_5,0.022271
211,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_15_ewma,0.019251
212,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_8_rolling_mean_20,0.016953
213,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_2_ewma,0.014738
214,FD002,xgboost,temporal,1b7584f00aad4746a59ff0103f2448c3,sensor_4_rolling_mean_10,0.012253


In [ ]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD003")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

,subset,model,feature_set,training_run_id,feature,importance
313,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_11_rolling_mean_5,0.220522
314,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_11_ewma,0.152208
315,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_3_rolling_mean_10,0.108075
316,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_17_ewma,0.104303
317,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_3_ewma,0.081463
318,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_9_rolling_mean_5,0.071893
319,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_17_rolling_mean_5,0.032501
320,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_17_rolling_mean_10,0.015322
321,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_11_rolling_mean_10,0.012077
322,FD003,xgboost,temporal,dd44b8093c5b44b495e5b7b38c5fce0c,sensor_12_rolling_slope_20,0.008445


In [27]:
feature_importance_results[(feature_importance_results.model=="xgboost") 
                           & (feature_importance_results.subset=="FD004")
                           & (feature_importance_results.feature_set=="temporal")].head(10)

,subset,model,feature_set,training_run_id,feature,importance
433,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_11_rolling_mean_5,0.247026
434,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_3_rolling_mean_10,0.132019
435,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_17_rolling_mean_10,0.077313
436,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_17_ewma,0.073489
437,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_14_rolling_mean_5,0.070691
438,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_9_rolling_mean_5,0.049667
439,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_11_ewma,0.038233
440,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_8_ewma,0.033568
441,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_3_ewma,0.026453
442,FD004,xgboost,temporal,1b90ed34b6964e58bcef186c090cfc52,sensor_9_ewma,0.024924
